### Capstone: Transaction Summary

**Purpose.** Read a messy CSV of crypto trades, clean it, build a structured summary, and save the result to a single text file.

**Pipeline.** `read` ➜ `clean` ➜ `summarize` ➜ `save`

#### 1. Input description

Source file: `transactions.csv`. Each row is one market trade with these columns:

`ID, Type, Subtype, Datetime, Amount, Amount currency, Value, Value currency, Rate, Rate currency`

The raw file has typical real-world noise:
- extra whitespace inside cells (`'475166337 '`, `' Buy '`)
- inconsistent casing (`'sell'`, `'SELL'`, `'Sell'`, `'btc'`, `'eur'`)
- one duplicated row (same `ID`)
- one row with missing numeric fields

The cleaning step has to handle all of this before any number is trusted.

In [ ]:
import csv
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from collections import Counter

DATA_PATH = Path('transactions.csv')
OUTPUT_PATH = Path('summary.txt')

NUMERIC_FIELDS = ['Amount', 'Value', 'Rate']
TEXT_FIELDS = ['Type', 'Subtype']
CURRENCY_FIELDS = ['Amount currency', 'Value currency', 'Rate currency']

#### 2. Read raw rows

`csv.DictReader` gives one dictionary per row using the header line as keys. We just collect them into a list to inspect.

In [14]:
def read_rows(path):
    with open(path, 'r', encoding='utf-8') as reader:
        return list(csv.DictReader(reader))

raw_rows = read_rows(DATA_PATH)
print(f'Raw rows: {len(raw_rows)}')
raw_rows[0]

Raw rows: 11


{'ID': '475166337 ',
 'Type': 'Market',
 'Subtype': ' Buy ',
 'Datetime': '2025-11-02T17:52:48Z',
 'Amount': '0.04500000',
 'Amount currency': 'BTC',
 'Value': '2812.50',
 'Value currency': 'EUR',
 'Rate': '62500',
 'Rate currency': 'EUR'}

- The strings are analized so we can see what errors are present in the data

##### Function `summary`

Goes through all rows and, for each cell that is not a number, counts how many times its value appears (after stripping whitespace). Returns a dictionary `{value: frequency}` of all text entries present in the dataset.

Goal: audit the categorical fields (`Type`, `Subtype`, `Amount currency`, `Value currency`, `Rate currency`) to detect inconsistencies before cleaning, such as:
- different spellings of the same value due to casing (`Buy`, `buy`, `Market`, `market`, `BTC`, `btc`)
- casing variations in `Subtype` (`Sell`, `sell`, `SELL`)
- empty strings revealing missing fields

This motivates the normalizations later applied in `clean_rows` (`title()` for text fields and `upper()` for currencies).

In [ ]:
def summary(rows):
    counts = Counter()
    for row in rows:
        for key, value in row.items():
            # Ignore numbers: try to convert to float, skip if successful
            try:
                float(value)
                continue
            except (ValueError, TypeError):
                pass
            counts[value.strip()] += 1
    return dict(counts)

resume = summary(raw_rows)

for key, value in resume.items():
    if "2025" in key:
        continue
    print(f"{key}, {value}")

Market, 10
Buy, 5
BTC, 10
EUR, 21
sell, 1
btc, 1
SELL, 1
Sell, 3
eur, 1
buy, 1
, 3
market, 1


##### Function `analize_ids`
 
Goes through the raw rows and classifies each `ID` into one of three categories: `unique`, `duplicate`, or `missing`. Returns a dictionary with the counts.
 
**Goal:** detect integrity problems in the identifier column of the dataset before cleaning, specifically:
- Empty or missing IDs, which indicate incomplete rows
- Repeated IDs, which reveal duplicate transactions that should be deduplicated

This justifies the rules later applied in `clean_rows`: discarding rows without required fields and keeping only the first occurrence of each ID.

In [23]:
def analize_ids(raw_rows):
    seen = set()
    counts = {'unique': 0, 'duplicate': 0, 'missing': 0}
    for row in raw_rows:
        id_ = row.get('ID', '').strip()
        if not id_:
            counts['missing'] += 1
        if id_ in seen:
            counts['duplicate'] += 1
        else:
            seen.add(id_)
            counts['unique'] += 1
    return counts

print(analize_ids(raw_rows))

{'unique': 10, 'duplicate': 1, 'missing': 0}


#### 3. Clean rows

Cleaning rules, applied in order:
1. strip whitespace from every field
2. drop rows whose `ID` was already seen (deduplicate, keep first)
3. drop rows with empty numeric fields
4. normalize text fields to title case and currencies to upper case
5. cast numeric fields to `float`
6. parse the ISO datetime string into a `datetime` object

In [16]:
def clean_rows(rows):
    seen_ids = set()
    cleaned = []
    for row in rows:
        item = {key: value.strip() for key, value in row.items()}
        if item['ID'] in seen_ids:
            continue
        if any(not item[field] for field in NUMERIC_FIELDS):
            continue
        for field in TEXT_FIELDS:
            item[field] = item[field].title()
        for field in CURRENCY_FIELDS:
            item[field] = item[field].upper()
        for field in NUMERIC_FIELDS:
            item[field] = float(item[field])
        item['Datetime'] = datetime.fromisoformat(item['Datetime'].replace('Z', '+00:00'))
        seen_ids.add(item['ID'])
        cleaned.append(item)
    return cleaned

clean = clean_rows(raw_rows)
print(f'Clean rows: {len(clean)} (dropped {len(raw_rows) - len(clean)})')
clean[0]

Clean rows: 9 (dropped 2)


{'ID': '475166337',
 'Type': 'Market',
 'Subtype': 'Buy',
 'Datetime': datetime.datetime(2025, 11, 2, 17, 52, 48, tzinfo=datetime.timezone.utc),
 'Amount': 0.045,
 'Amount currency': 'BTC',
 'Value': 2812.5,
 'Value currency': 'EUR',
 'Rate': 62500.0,
 'Rate currency': 'EUR'}

#### 4. Summarize

For each `Subtype` (Buy / Sell) we compute count, total amount in BTC, total value in EUR, and average price (EUR per BTC). We also report the date range and the net BTC position.

In [17]:
def summarize(rows):
    by_subtype = defaultdict(lambda: {'count': 0, 'total_amount': 0.0, 'total_value': 0.0})
    for row in rows:
        bucket = by_subtype[row['Subtype']]
        bucket['count'] += 1
        bucket['total_amount'] += row['Amount']
        bucket['total_value'] += row['Value']
    for bucket in by_subtype.values():
        bucket['avg_price'] = bucket['total_value'] / bucket['total_amount'] if bucket['total_amount'] else 0.0
    dates = [row['Datetime'] for row in rows]
    return {
        'total_transactions': len(rows),
        'date_range': (min(dates), max(dates)),
        'by_subtype': dict(by_subtype),
        'net_btc': by_subtype['Buy']['total_amount'] - by_subtype['Sell']['total_amount'],
    }

summary = summarize(clean)
summary

{'total_transactions': 9,
 'date_range': (datetime.datetime(2025, 11, 2, 17, 52, 48, tzinfo=datetime.timezone.utc),
  datetime.datetime(2025, 11, 10, 10, 21, 12, tzinfo=datetime.timezone.utc)),
 'by_subtype': {'Buy': {'count': 4,
   'total_amount': 0.15,
   'total_value': 9462.65,
   'avg_price': 63084.333333333336},
  'Sell': {'count': 5,
   'total_amount': 0.11374999999999999,
   'total_value': 7313.0,
   'avg_price': 64290.109890109896}},
 'net_btc': 0.036250000000000004}

#### 5. Save the result

We turn the summary dict into a readable plain-text report and write it to `summary.txt`. One file in, one file out.

In [18]:
def format_summary(summary):
    start, end = summary['date_range']
    lines = [
        'Transactions summary',
        '=' * 30,
        '',
        f"Total transactions: {summary['total_transactions']}",
        f'Date range: {start.date()} to {end.date()}',
        f"Net BTC position: {summary['net_btc']:.8f} BTC",
        '',
        'By subtype',
        '-' * 30,
    ]
    for subtype, bucket in sorted(summary['by_subtype'].items()):
        lines.append(f'{subtype}:')
        lines.append(f"  count:        {bucket['count']}")
        lines.append(f"  total amount: {bucket['total_amount']:.8f} BTC")
        lines.append(f"  total value:  {bucket['total_value']:.2f} EUR")
        lines.append(f"  avg price:    {bucket['avg_price']:.2f} EUR/BTC")
    return '\n'.join(lines)

def save_summary(path, text):
    with open(path, 'w', encoding='utf-8') as writer:
        writer.write(text)

report = format_summary(summary)
save_summary(OUTPUT_PATH, report)

with open(OUTPUT_PATH, 'r', encoding='utf-8') as reader:
    print(reader.read())

Transactions summary

Total transactions: 9
Date range: 2025-11-02 to 2025-11-10
Net BTC position: 0.03625000 BTC

By subtype
------------------------------
Buy:
  count:        4
  total amount: 0.15000000 BTC
  total value:  9462.65 EUR
  avg price:    63084.33 EUR/BTC
Sell:
  count:        5
  total amount: 0.11375000 BTC
  total value:  7313.00 EUR
  avg price:    64290.11 EUR/BTC


#### 6. Result

Output file: `summary.txt`. It contains the trade count, date range, net BTC position, and per-subtype totals and average price. The notebook is idempotent: rerun from top to bottom and it always produces the same `summary.txt` from the same `transactions.csv`.